# 🚀 Kiara Foundation Model — Cloud GPU Pre-Training (Google Colab / Kaggle)

This notebook trains the **Kiara-Small (~125M parameter)** foundation model on free cloud GPUs (NVIDIA T4 or A100) using **PyTorch Mixed Precision (AMP)**.

### Why train here?
- **50x Faster**: ~15–20 minutes on a free T4 GPU vs 15+ hours on a laptop CPU.
- **Zero Laptop Wear**: Your laptop stays cool, safe, and silent.

## 1. Verify Cloud GPU Acceleration

In [1]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ Warning: GPU is not active. Go to Runtime -> Change runtime type -> Select T4 GPU.")

Thu Oct 15 08:45:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive (To Persist Trained Checkpoints)

In [2]:
from google.colab import drive
import os

drive.mount('/content/drive')
CHECKPOINT_BACKUP_DIR = '/content/drive/MyDrive/kiara_checkpoints'
os.makedirs(CHECKPOINT_BACKUP_DIR, exist_ok=True)
print(f"Checkpoints will be automatically backed up to: {CHECKPOINT_BACKUP_DIR}")

Mounted at /content/drive
Checkpoints will be automatically backed up to: /content/drive/MyDrive/kiara_checkpoints


## 3. Clone Kiara Repository & Install Dependencies

In [1]:
# Clone the repository (or pull latest updates)
%cd /content
!git clone https://github.com/varunvj151/kiara.git || (cd kiara && git pull)
%cd /content/kiara

# Install dependencies
!pip install -q -r requirements.txt

/content
Cloning into 'kiara'...
remote: Enumerating objects: 286, done.
remote: Counting objects: 100% (286/286), done.
remote: Compressing objects: 100% (209/209), done.
remote: Total 286 (delta 70), reused 273 (delta 62), pack-reused 0 (from 0)
Receiving objects: 100% (286/286), 14.43 MiB | 13.34 MiB/s, done.
Resolving deltas: 100% (70/70), done.
/content/kiara
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.0/258.0 kB 12.5 MB/s eta 0:00:00


## 4. Run Pre-Flight Smoke Test (Verifies GPU Tensors & Backprop in < 5 seconds)

In [2]:
!python experiments/smoke_test.py


  KIARA PRE-FLIGHT SMOKE TEST (Low-Intensity Thermal-Safe)
  PyTorch: 2.11.0+cu130 | Active Threads: 4

[1/5] Testing Tokenizer...
  Tokenizer OK! Vocab size: 32,768

[2/5] Initializing Test Model...
  Model instantiated cleanly! Parameters: 10,750,208

[3/5] Creating Test Mini-Batch (16 steps)...

[4/5] Executing 16-Step Verification Loop...
  Step  4/16 | Loss: 9.9933 | LR: 4.78e-04
  Step  8/16 | Loss: 9.2408 | LR: 3.25e-04
  Step 12/16 | Loss: 8.8511 | LR: 1.35e-04
  Step 16/16 | Loss: 8.6031 | LR: 5.00e-05

[5/5] Testing Checkpoint Save & Load...
  Checkpoint serialization verified!

  SUCCESS: All 5 Verification Gates Passed in 10.7s!
  Loss reduced: 10.4300 -> 8.6031
  System status: GREEN. Architecture is 100% verified for Cloud GPU training.



## 5. Launch Stage 26D.1 Pre-Training (Kiara-Small 125M)

In [3]:
# Execute Stage 26D.1 training
!python experiments/pretrain_26D1.py


# Backup checkpoints to Google Drive
!cp -r checkpoints/stage_26D1/* /content/drive/MyDrive/kiara_checkpoints/
print("✅ Training complete! Checkpoints safely saved in Google Drive.")


  STAGE 26D.1: PARAMETER CAPACITY SCALING  (Kiara-Tiny 40.9M -> Kiara-Small ~125M)
  Corpus: Kiara-Real-v2.0 [FROZEN] | Context: 64T [FROZEN] | Steps: 3,000 [FROZEN]

/content/kiara/experiments/pretrain_26D1.py:91: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
  Training Device: cuda (AMP Enabled: True)
  Model: Kiara-Small  |  Total params: 99,502,848
  Architecture: d_model=768, layers=12, heads=12Q/3KV
  Packaging 28,632 training docs (seq_len=64)...
  Train chunks: 102,115 | Eval chunks: 11,503

  [Step 0 Baseline] Eval Loss: 10.5461  PPL: 38029.2

    Step   Train Loss   Train PPL   Eval Loss    Eval PPL          LR      Tok/s       ETA
------------------------------------------------------------------------------------------
/content/kiara/experiments/pretrain_26D1.py:136: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please us

## 6. Test Model Generation

In [4]:
import torch
from kiara.model import KiaraConfig, KiaraForCausalLM
from kiara.tokenizer import KiaraTokenizer
from kiara.trainer import load_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = KiaraTokenizer.load()
config = KiaraConfig.from_pretrained("checkpoints/stage_26D1")
model = KiaraForCausalLM(config).to(device)

load_checkpoint("checkpoints/stage_26D1/checkpoint_final.pt", model=model)
model.eval()

prompt = "The key mechanism in transformer architectures"
input_ids = torch.tensor([tokenizer.encode(prompt)], device=device)

with torch.no_grad():
    for _ in range(30):
        logits = model(input_ids).logits[:, -1, :]
        next_tok = torch.argmax(logits, dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_tok], dim=1)

output_text = tokenizer.decode(input_ids[0].tolist())
print("\n--- Model Output ---")
print(output_text)


--- Model Output ---
The key mechanism in transformer architectures.[EOS]# variant 5333
def batch_encode(texts, tokenizer, max_length=


In [5]:
!python experiments/pretrain_26D2.py --seq-len 256



  STAGE 26D.2: CONTEXT WINDOW SCALING (T = 256 tokens, Batch = 8)
  Model: Kiara-Tiny (40.9M) [FROZEN] | Corpus: Kiara-Real-v2.0 [FROZEN] | Steps: 3,000
  Tokens/step: 2,048 | Total Tokens: 6,144,000 [FROZEN]

  Training Device: cuda
  Verified Model Params: 40,903,168 (FROZEN 40.9M)
  Packaging 28,632 training docs (seq_len=256)...
  Train chunks: 25,528 | Eval chunks: 2,875

  [Step 0 Baseline] Multi-Domain Eval Loss: 10.5035 (PPL: 36441.4)

    Step   Train Loss   Train PPL   Eval Loss    Eval PPL          LR      Tok/s       ETA
------------------------------------------------------------------------------------------
     250       5.8115      334.13      2.6860       14.67    2.99e-04     9405.8  00:09:58
     500       3.8412       46.58      1.5997        4.95    2.90e-04     9489.0  00:08:59
  [ckpt] Saved checkpoint_step_500.pt
     750       3.0179       20.45      1.3736        3.95    2.72e-04     9422.9  00:08:09
    1000       2.5606       12.94      1.2541        3.50 

In [6]:
%cd /content/kiara
!git pull


/content/kiara
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 13 (delta 5), reused 13 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (13/13), 30.73 KiB | 449.00 KiB/s, done.
From https://github.com/varunvj151/kiara
   1e65284..f77c99a  main       -> origin/main
Updating 1e65284..f77c99a
Fast-forward
 data/instructions/eval_sft.jsonl          |  182 ++++
 data/instructions/generate_sft_dataset.py |  207 ++++
 data/instructions/train_sft.jsonl         | 1634 +++++++++++++++++++++++++++++
 experiments/finetune_sft.py               |  300 ++++++
 notebooks/kiara_colab_trainer.ipynb       |   39 +
 scripts/chat.py                           |   28 +-
 6 files changed, 2385 insertions(+), 5 deletions(-)
 create mode 100644 data/instructions/eval_sft.jsonl
 create mode 100644 data/instructions/generate_sft_dataset.py
 create mode 100644 data/instructions/train_sft.jsonl
 create mode 10

In [7]:
!python experiments/finetune_sft.py --base-ckpt checkpoints/stage_26D2_256/checkpoint_final.pt --steps 800 --batch-size 8 --lr 1e-4

# Backup instruct checkpoint to Google Drive
!cp -r checkpoints/kiara_instruct/* /content/drive/MyDrive/kiara_checkpoints/
print("✅ Kiara-Instruct model trained and safely backed up to Google Drive!")


  KIARA SUPERVISED INSTRUCTION FINE-TUNING (SFT)
  Base Model Checkpoint: checkpoints/stage_26D2_256/checkpoint_final.pt
  Target Steps: 800 | Batch Size: 8 | Seq Len: 256 | LR: 0.0001
/content/kiara/experiments/finetune_sft.py:152: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_cuda)
  Device: cuda | Mixed Precision (AMP): True

Loading base weights from checkpoints/stage_26D2_256/checkpoint_final.pt...
  Tokenizing and masking instruction datasets...
  Train samples: 1,634 | Eval samples: 182

  [Step 0 Pre-SFT Baseline] Eval Loss: 6.1625 (PPL: 474.61)
  [Step 0 Baseline Answer to 'Who are you?']: '"model_type": "
def load_checkpoint(path, model, optimizer=None, scheduler=None, device="cpu"):
 ckpt = torch.load(path, map_location=de'

    Step   Train Loss   Train PPL   Eval Loss    Eval PPL          LR       ETA
--------------------------------------------

In [8]:
!python scripts/chat.py --checkpoint checkpoints/kiara_instruct/checkpoint_final.pt --instruct --prompt "Who are you?"
!python scripts/chat.py --checkpoint checkpoints/kiara_instruct/checkpoint_final.pt --instruct --prompt "Write a Python function to check if a number is prime."


Device: cuda | Active CPU Threads: 4 | Instruct Mode: True
Loading checkpoint from: checkpoints/kiara_instruct/checkpoint_final.pt...
Kiara Ready! Parameters: 40,903,168

Prompt: Who are you?
Generated: I am Kiara, an intelligent, reasoning-first AI foundation model built from first
       .
[43 tokens, 54.1 tok/s]

Device: cuda | Active CPU Threads: 4 | Instruct Mode: True
Loading checkpoint from: checkpoints/kiara_instruct/checkpoint_final.pt...
Kiara Ready! Parameters: 40,903,168

Prompt: Write a Python function to check if a number is prime.
Generated: def is_palindrome(s: str) -> bool:
    cleaned = ''.join(c.lower() for c in s if c.isalnum())
    return cleaned ==
[60 tokens, 58.8 tok/s]

